Step 1: Bootstrap & install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
!pip install --quiet langgraph langgraph-checkpoint-sqlite
!pip install --quiet langchain_core
!pip install fastapi uvicorn -q


from google.colab import userdata, drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')


import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 2: Set up env vars and then import

In [ ]:
import os
from google.colab import userdata, drive


# Clear any lingering LangChain-prefixed vars from earlier attempts
for old in ("LANGCHAIN_TRACING_V2", "LANGCHAIN_API_KEY", "LANGCHAIN_PROJECT", "LANGCHAIN_ENDPOINT"):
    os.environ.pop(old, None)

# Set only the LangSmith-prefixed vars
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY").strip()
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["LANGSMITH_PROJECT"] = "astra-swarm-week5"

from astra_swarm import graph

# LangSmith Service Keys start with lsv2_sk_ or ls__ (varies by generation) and are 40–70 characters. If prefix is lsv2_pt_, that's a Personal Access Token, not a Service Key

import os
key = os.environ.get("LANGSMITH_API_KEY", "")
print(f"Key set: {bool(key)}")
print(f"Key length: {len(key)}")
print(f"Key prefix: {key[:8] if key else 'EMPTY'}...")
print(f"Key suffix: ...{key[-4:] if key else 'EMPTY'}")

Step 3: Test the interrupt flow

In [ ]:
import json
from pathlib import Path
from astra_swarm.graph import triage_graph, new_incident_state
from langgraph.types import Command
from langchain_core.runnables import RunnableConfig

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/20_alerts.json").read_text()
)

for alert in alerts:
    print(alert)


# High-severity alert that will trigger the approval gate
alert = alerts[1]  # pick one you know escalates
state = new_incident_state(alert)
config: RunnableConfig = {"configurable": {"thread_id": "hitl-demo-1"}}

# First invocation — runs until interrupt fires
result = triage_graph.invoke(state, config=config)

# Check whether we hit an interrupt
if "__interrupt__" in result:
    interrupt_data = result["__interrupt__"][0].value
    print(f"[PAUSED] Graph interrupted for approval:")
    print(f"  Incident: {interrupt_data['incident_id']}")
    print(f"  Reason: {interrupt_data['reason']}")
    print(f"  Workers run: {interrupt_data['workers_run']}")
    print(f"  Summary: {interrupt_data['investigation_summary'][:200]}")

    # Simulate operator approval
    resume_command = Command(resume={"approved": True, "note": "Confirmed by SOC lead"})
    final_result = triage_graph.invoke(resume_command, config=config)
    print(f"\n[RESUMED] Final state: escalated={final_result.get('escalated')}")
else:
    print("[NOT INTERRUPTED] Alert did not trigger escalation path")

Step 4: Local test approach with env vars

In [ ]:
import subprocess
import os

# First, kill any prior uvicorn to avoid port conflicts
os.system("pkill -f 'uvicorn.*astra_swarm' 2>/dev/null; sleep 1")

# Set PYTHONPATH so uvicorn can import astra_swarm
env = os.environ.copy()
env["PYTHONPATH"] = "/content/astra-swarm/src"
# Preserve your API keys in the subprocess env
env["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY", "")
env["LANGSMITH_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "")
env["LANGSMITH_TRACING"] = "true"
env["LANGSMITH_PROJECT"] = "astra-swarm-week6-api"

log_file = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(
    ["uvicorn", "astra_swarm.api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,                      # ← key line
    start_new_session=True,
)
print(f"Started uvicorn PID {proc.pid}")

# Wait for readiness
import requests, time
for i in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).status_code == 200:
            print(f"✓ Ready after {i+1}s")
            break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("✗ Not ready. Log:")
    print(open("/tmp/uvicorn.log").read())

Step 4: Local test approach 2 with directory

In [ ]:
import subprocess
import os

# First, kill any prior uvicorn to avoid port conflicts
os.system("pkill -f 'uvicorn.*astra_swarm' 2>/dev/null; sleep 1")

!cd /content/astra-swarm && pip install -e . -q

log_file = open("/tmp/uvicorn.log", "w")
proc = subprocess.Popen(
    ["uvicorn", "astra_swarm.api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,                      # ← key line
    start_new_session=True,
)
print(f"Started uvicorn PID {proc.pid}")

# Wait for readiness
import requests, time
for i in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).status_code == 200:
            print(f"✓ Ready after {i+1}s")
            break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("✗ Not ready. Log:")
    print(open("/tmp/uvicorn.log").read())

Step 5: Verify if the thread is alive and do Healthcheck.

In [ ]:
print(f"Thread alive: {t.is_alive() if 't' in dir() else 'thread variable missing'}")

# Try to reach the port with a socket check (no HTTP layer)
import socket
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(2)
result = sock.connect_ex(('127.0.0.1', 8000))
sock.close()
print(f"Port 8000 open: {result == 0}  (result code: {result})")

import requests
import time

for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if r.status_code == 200:
            print(f"✓ API ready after {i+1}s")
            break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("✗ API did not start within 30s")

Step 6: Final test of triaging low-severity and high-severity alert.

In [ ]:
import requests

# Health check first
r = requests.get("http://127.0.0.1:8000/health")
print(f"Health: {r.status_code} {r.json()}")

# Triage a low-severity alert (should complete without pausing)
low_sev = "Firewall: connection dropped from 10.0.0.5 to 10.0.0.6:8080. Standard block."
r = requests.post("http://127.0.0.1:8000/triage", json={"alert": low_sev})
print(f"\nTriage: {r.status_code}")
print(f"Status: {r.json().get('status')}")
print(f"Workers: {r.json().get('workers_run')}")

# Triage a high-severity alert (should pause for approval)
high_sev = "CEF:0|CrowdStrike|Falcon|7.11.0|malware-detection|Malware Detected|9|suser=alice src=10.0.0.5 fname=trojan.exe severity=high"
r = requests.post("http://127.0.0.1:8000/triage", json={"alert": high_sev})
data = r.json()
print(f"\nHigh-sev triage: {r.status_code}")
print(f"Status: {data.get('status')}")
if data.get('status') == 'awaiting_approval':
    print(f"Prompt: {data.get('approval_prompt')}")

    # Approve
    r2 = requests.post("http://127.0.0.1:8000/approve", json={
        "thread_id": data['thread_id'],
        "approved": True,
        "note": "Testing approve path",
    })
    print(f"\nFinal after approval: {r2.json().get('status')}, escalated={r2.json().get('escalated')}")